## Overview
This notebook implements a complete pipeline for analyzing an event log and deriving rule-based interpretable explanations of the trace variants.

The workflow includes:
- Data Loading  
- Trace Extraction
- Variants Analysis
- Data Encoding
- Trace Clustering
- Decision Tree Realization
- Decision Table Construction
- Rules Extraction
- Rules Translation

## Data Loading
This section imports all the required libraries and loads the event log from a XES file.
The event log is represented in a structured format where each case is identified by a unique `case_id` and each event contains attributes such as the activity name (`concept:name`) and timestamp (`time:timestamp`).

In [ ]:
#Imports
from pm4py.objects.log.importer.xes import importer as xes_importer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import re
import math
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.tree import _tree
from sklearn.metrics import silhouette_score
from collections import Counter
from IPython.display import display, HTML

#Pandas display options
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

In [ ]:
#event Log loading
event_log = xes_importer.apply("nameOfTheDataset.xes") #to test the code instead of nameOfTheDataset insert the actual name of the dataset (like order, sepsis_10 or hospital_12)

## Trace Extraction
In this section, we extract individual traces from the loaded event log. 
Each trace corresponds to a single case in the process and is represented 
as an ordered sequence of activities. 

For each trace, the sequence of activity names is preserved to maintain 
the chronological flow of the process. The `case_id` is also stored 
for reference in later steps.

In [ ]:
#initialize storage for sequences
sequences = []

#iterate over all traces in the event log
for i, trace in enumerate(event_log):
    case_id = trace.attributes['concept:name']
    print(f"Trace {i} (case_id: {case_id}):")

    #extract sequence of activity names
    trace_sequence = [event['concept:name'] for event in trace]
    sequences.append(trace_sequence)
    
    #print each event with its timestamp
    for event in trace:
        print(f"  {event['concept:name']} at {event['time:timestamp']}")
    print("___________________________________")

## Variants Analysis
Once traces have been extracted, the next step is to identify unique trace variants.
A trace variant represents a distinct sequence of activities in the process. 

For each trace, we replace spaces in activity names with underscores to standardize the labels and then identify all unique sequences. Each unique sequence is assigned a Variant ID. This allows each trace to be unambiguously linked to a specific variant.

In [ ]:
#initialize lists for variants identification
all_transitions = []
all_activities = []
case_ids = []
variants = set()

#iterate over all traces
for trace in event_log:
    case_id = trace.attributes['concept:name']
    case_ids.append(case_id)
    
    #sequence of activity names (standardized)
    events = [event['concept:name'].replace(' ', '_') for event in trace]
    all_activities.append(events)
    
    #transitions between activities
    transitions = [f"{events[i]}-->{events[i+1]}" for i in range(len(events)-1)]
    all_transitions.append(transitions)
    
    #add the unique sequence to the variants set
    variants.add(tuple(events))

#create a dataframe of traces with transitions
transitions_df = pd.DataFrame({
    'CASE_ID': case_ids,
    'SEQUENCE': all_activities,
    'TRANSITIONS': all_transitions
})
transitions_df.set_index('CASE_ID', inplace=True)

#create the dataframe of the variants
variants_sorted = sorted(list(variants))

variant_rows = []
for i, seq in enumerate(variants_sorted, start=1):
    variant_rows.append({
        'Variant': i,
        'Sequence': list(seq)
    })

variants_df = pd.DataFrame(variant_rows)
variants_df

## Data Encoding
In this section, each trace is encoded into a structured, machine-readable format. 
Both activities and transitions are transformed into binary features using a one-hot-like encoding approach. 

Each unique activity and each unique transition becomes a feature: a value of `1` indicates that it occurs in the trace, while `0` indicates its absence. 
This representation preserves the behavioral patterns of traces while allowing 
systematic analysis.


In [ ]:
#Encoding for the transitions

#convert each trace's transitions into a single string separated by spaces and include only traces that have at least one transition
transition_strings = [' '.join(trans) for trans in all_transitions if len(trans) > 0]

#keep only the corresponding case IDs for traces that have transitions
filtered_case_ids = [cid for cid, trans in zip(case_ids, all_transitions) if len(trans) > 0]

vectorizer_trans = CountVectorizer(token_pattern=r'[^ ]+') #initialize CountVectorizer to create one-hot-like encoding for transitions
X_trans = vectorizer_trans.fit_transform(transition_strings) #fit and transform the transition strings into a binary feature matrix

#convert the matrix into a DataFrame for easier handling
df_transitions = pd.DataFrame(
    X_trans.toarray(),
    columns=vectorizer_trans.get_feature_names_out(),
    index=filtered_case_ids
)

df_transitions = df_transitions.reindex(case_ids, fill_value=0)


#Encoding for the activities

#convert each trace's activities into a single string separated by spaces
activity_strings = [' '.join(activity) for activity in all_activities]

vectorizer_act = CountVectorizer(token_pattern=r'[^ ]+') #initialize CountVectorizer for activities
X_act = vectorizer_act.fit_transform(activity_strings) #fit and transform activity strings into a binary feature matrix

#convert the activity matrix into a DataFrame
df_activities = pd.DataFrame(
    X_act.toarray(),
    columns=vectorizer_act.get_feature_names_out(),
    index=case_ids
)

#combine activity and transition features
encoded_data = pd.concat([df_activities, df_transitions], axis=1)
encoded_data.head()

## Trace Clustering
Once traces have been encoded, the next step is to group them according to 
their behavioral similarity. 

The encoded activity and transition features are used to represent each trace as a numerical vector. To determine the appropriate number of clusters, the silhouette score is first computed over a small range of candidate values around the number of trace variants. The value of k that maximizes the silhouette score is then selected, which in our case is observed to correspond to the number of variants. K-Means clustering is subsequently applied to partition the traces into k clusters, with each cluster ideally corresponding to a distinct trace variant.

In [ ]:
#convert the encoded data to a numeric array
x = encoded_data.values

#evaluate the silhouette scores for a small range around the number of variants
sil_scores = []
num_variants = len(variants)
margin = 2
k_range = range(max(2, num_variants - margin), num_variants + margin)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(x)
    score = silhouette_score(x, labels)
    sil_scores.append(score)
    print(f"k={k}, silhouette score={score:.4f}")

#select the best number of clusters which should be equivalent to the number of variants
best_k = k_range[sil_scores.index(max(sil_scores))]
print(f"The best k is {best_k}")

In [ ]:
#fit the K-Means with the best number of clusters
x = encoded_data.values
kmeans_best = KMeans(n_clusters=best_k, random_state=42)
encoded_data['Cluster'] = kmeans_best.fit_predict(x)

## Decision Tree Realization
Once traces have been clustered, a decision tree is trained to model the relationship between trace features (activities and transitions) and cluster assignments. 

The input features consist of the binary representations of activities and transitions, while the cluster labels obtained from K-Means serve as the target variable. 

The decision tree allows us to identify which features are most relevant in determining cluster membership. Feature importance values quantify the contribution of each activity or transition in predicting the cluster of a trace. Higher values indicate stronger influence on cluster assignment, highlighting which elements of the process are critical for distinguishing trace variants.

In [ ]:
#define the features (X) and the target (y)
X = encoded_data.drop(columns='Cluster')
y = encoded_data['Cluster']

#train the decision tree
classifier = DecisionTreeClassifier(random_state=42)
classifier.fit(X, y)

#display the feuture importances
importances = classifier.feature_importances_
indices = np.argsort(importances)[::-1]

for name, importance in zip(X.columns, importances):
    print(f"Feature Importance: [{name} : {importance:.3f}]")

#plot the decision tree
plt.figure(figsize=(15,10))
plot_tree(
    classifier,
    feature_names=X.columns,
    class_names=[str(c) for c in np.unique(y)],
    filled=True,
    fontsize=10
)
plt.show()

## Decision Table Construction
In this step, the decision tree is converted into a decision table to represent the classification rules in a structured, tabular format.

Before constructing the table, each cluster is mapped to its corresponding trace variants based on the activity sequences. This ensures that the decision table references the actual trace variants rather than generic cluster IDs.

Then, the table is constructed: each row corresponds either to a condition on a feature (activity or transition) or an action (the trace variant corresponding to the cluster assignment), while each column corresponds to a distinct rule derived from a path from the root to a leaf of the decision tree.

This tabular representation provides a clear overview of the rules learned by the tree and serves as the basis for extracting interpretable trace variant rules.

In [ ]:
#join activities of each trace into a single string
trace_sequences = ['|'.join(seq) for seq in all_activities]

#create the dataframe with cluster assignments
df_trace_cluster = pd.DataFrame({
    'CASE_ID': case_ids,
    'Sequence': trace_sequences,
    'Cluster': encoded_data['Cluster']
})

#create a mapping from sequence to Variant_ID
variant_dict = {'|'.join(var['Sequence']): var['Variant'] for _, var in variants_df.iterrows()}

#map Variant_ID for each trace
df_trace_cluster['Variant_ID'] = df_trace_cluster['Sequence'].map(variant_dict)

#check for traces that could not be mapped (it's just a security check)
if df_trace_cluster['Variant_ID'].isna().any():
    print("Warning: alcune trace non hanno Variant_ID!")

#group variants by cluster
variants_per_cluster = df_trace_cluster.groupby('Cluster')['Variant_ID'].unique().apply(sorted)

#print the variants present in each cluster which, in this work, will be one variant per cluster
for cluster, variants in variants_per_cluster.items():
    variants_list = [int(v) for v in variants]
    print(f"Cluster {cluster} has variants: {sorted(variants_list)}")

In [ ]:
def decision_tree_to_table(classifier, X, y):
    tree_ = classifier.tree_ # internal structure of the decision tree: nodes, leaves, thresholds
    
    #build a list of class names based on the variants in each cluster
    class_names = []
    for c in np.unique(y):
        variants_in_cluster = variants_per_cluster[c]
        class_names.extend([f"Variant {int(v)}" for v in variants_in_cluster])

    paths = []

    #recursive function to collect conditions along each path from root to leaf
    def save_conditions(node, conditions):
        if tree_.feature[node] != _tree.TREE_UNDEFINED: #internal node
            name = X.columns[tree_.feature[node]] # feature used for splitting
            threshold = tree_.threshold[node]

            condition_name = f"{name} <= {threshold:.2f}"

            #traverse left child (condition True)
            save_conditions(tree_.children_left[node], conditions + [(condition_name, True)]) #it basically follows the left child of the node where the condition is true and it adds it to the list of the path
            #traverse right child (condition False)
            save_conditions(tree_.children_right[node], conditions + [(condition_name, False)])
        else:
            #leaf node: save the path and predicted class (class with most samples)
            paths.append((conditions, tree_.value[node].argmax())) #it saves the path which is in conditions and it also saves the predicted class which is the one with most samples. Because tree_.value contains the number of samples for each class in the leaf

    save_conditions(0, [])

    #collect all unique conditions across all paths
    all_conditions = []
    for conditions, _ in paths: #consider only conditions, ignore class indices
        for condition_name, _ in conditions:
            if condition_name not in all_conditions:
                all_conditions.append(condition_name)

    #prepare rows for the decision table: IF/AND for conditions
    rows_type = [] #'IF', 'AND', 'THEN'
    rows_name = [] #names of conditions and actions
    for i, c in enumerate(all_conditions):
        rows_type.append('IF' if i == 0 else 'AND') 
        rows_name.append(c)
    for i, c in enumerate(class_names):
        rows_type.append('THEN' if i == 0 else 'AND')
        rows_name.append(c)

    table = {' ': rows_type, 'Conditions/Actions': rows_name}

    #fill each rule column with T/F for conditions and X/- for actions
    for idx, (conditions, class_idx) in enumerate(paths):
        rule_column = []
        cond_dict = {c: '-' for c in all_conditions} #default: '-' for conditions not present
        for c_name, val in conditions:
            cond_dict[c_name] = 'T' if val else 'F'
        for c in all_conditions:
            rule_column.append(cond_dict[c])
        
        #actions: all variants corresponding to the predicted cluster
        predicted_cluster = class_idx  
        variants_for_cluster = [f"Variant {int(v)}" for v in variants_per_cluster[predicted_cluster]]

        for c in class_names:
            rule_column.append('X' if c in variants_for_cluster else '-')

        table[f'Rule {idx+1}'] = rule_column

    df = pd.DataFrame(table)
    return df

#generate the decision table
decision_table = decision_tree_to_table(classifier, X, y)
decision_table

## Rules Extraction
After constructing the decision table, the next step is to systematically extract the rules that describe how traces are assigned to trace variants.

Each rule corresponds to a unique path from the root to a leaf in the decision tree and is represented in the decision table as a combination of feature conditions (activities or transitions) and the resulting action (the trace variant corresponding to the cluster assignment).

The extraction process converts each column of the decision table into an explicit IF–THEN statement, clearly specifying which conditions must hold for a trace to be assigned to a specific trace variant.

In [ ]:
def extract_rules_from_table(df):
    rules = []

    #select only the columns that correspond to rules
    rule_columns = [col for col in df.columns if col.startswith("Rule")]

    #identify the row where 'THEN' starts (all rows before are conditions)
    then_rows = df[df[' '] == 'THEN']
    then_start = then_rows.index[0]

    #iterate over each rule column
    for column in rule_columns:
        conditions = []

        #extract all condition rows (before 'THEN')
        for idx, row in df.iloc[:then_start].iterrows(): 
            val = row[column] #e.g., 'T', 'F', or '-'
            if val != '-': #only T or F are considered as conditions
                condition_str = f"{row['Conditions/Actions']} == {val}"
                conditions.append(condition_str)

        #extract action rows (rows starting from 'THEN')
        actions = []
        for idx, row in df.iloc[then_start:].iterrows(): 
            if row[column] == 'X': #if this action applies to the rule
                actions.append(row['Conditions/Actions']) 

        #combine all conditions into a single IF string
        if conditions:
            if_str = " AND ".join(conditions)
        else:
            if_str = "True" #default if no conditions

        #combine all actions into a single THEN string
        then_str = ", ".join(actions)
        
        #create the final IF–THEN rule string
        rule_str = f"{column}: IF {if_str} THEN {then_str}"
        rules.append(rule_str)

    return rules

#generate rules from the decision table
rules = extract_rules_from_table(decision_table)

for r in rules:
    print(r)

## Rules Translation
The final step of the pipeline translates the extracted rules into **human-understandable explanations**. 

The rules, which formally describe trace variant membership with conditions on features (activities and transitions), are converted into readable sentences that explain why a trace is assigned to a specific variant.

This translation process includes:

- Mapping feature identifiers into readable activity or transition names  
- Interpreting numerical thresholds and truth values to semantic statements  
- Handling self-loops and repeated activities  
- Highlighting the most decisive conditions using a green color scale, where rarer and more discriminative conditions are shown in darker shades  

The output is displayed in HTML format, providing an intuitive visualization for analysts.

In [ ]:
#function to capitalize words and replace underscores with spaces
def clean_activity_name(name):
    return " ".join(word.capitalize() for word in name.split("_"))

In [ ]:
#function to convert a number into human-readable "time/times"
def number_to_times(n):
    return f"{n} time" if n == 1 else f"{n} times"

In [ ]:
#function to identify if feature is an activity, a transition or a self-loop
def describe_feature(feature):
    if "-->" in feature:
        src, tgt = feature.split("-->")
        src_clean = clean_activity_name(src)
        tgt_clean = clean_activity_name(tgt)
        if src == tgt:
            return {"type": "self_loop", "activity": src_clean, "text": f"the activity '{src_clean}'"}
        else:
            return {"type": "transition", "activity": f"{src_clean} -> {tgt_clean}", 
                    "text": f"the transition from activity '{src_clean}' to activity '{tgt_clean}'"}
    else:
        act = clean_activity_name(feature)
        return {"type": "activity", "activity": act, "text": f"the activity '{act}'"}

In [ ]:
#function to translate a single rule into a semantic expression
def explain_condition(feature, operator, threshold, truth_value):
    threshold = float(threshold)
    truth_value = truth_value == "T"  #T=True, F=False
    info = describe_feature(feature)
    explanation = {}

    #self-loops
    if info["type"] == "self_loop" and operator == "<=":
        N = int(math.floor(threshold))
        if truth_value:
            #occurs at most N times
            explanation["self_loop_min"] = 0
            explanation["self_loop_max"] = N
        else:
            #occurs more than N times
            explanation["self_loop_min"] = N + 1
            explanation["self_loop_max"] = None

    #transition presence (threshold 0.5)
    elif info["type"] == "transition" and operator == "<=" and abs(threshold - 0.5) < 1e-6:
        explanation["transition_occurs"] = truth_value 

    #activity presence (threshold 0.5)
    elif info["type"] == "activity" and operator == "<=" and threshold <= 0.5:
        explanation["activity_occurs"] = truth_value 

    #generic activity counts
    elif info["type"] == "activity" and operator == "<=":
        N = int(math.floor(threshold))
        if truth_value:
            explanation["count_max"] = N
        else:
            explanation["count_min"] = N + 1

    return info["activity"], info["text"], explanation

In [ ]:
#colormaps for visualizing rule conditions: green shades represent presence (like "does occur", "occurs at least", "occurs at most N times",..) while the red shades represent absence (like "does not occur", "is never repeated")
#instead the intensity of the color encodes the frequency of the condition across the rules
green_cmap = matplotlib.colormaps["Greens"]
red_cmap = matplotlib.colormaps["Reds"]

def color_selector(freq, min_freq, max_freq, is_positive=True):
    #normalize frequency to [0,1] for color intensity scaling
    if max_freq == min_freq:
        normalized = 0.5
    else:
        normalized = 1 - (freq - min_freq) / (max_freq - min_freq)

    #select colormap based on semantic meaning
    cmap = green_cmap if is_positive else red_cmap
    r, g, b, _ = cmap(normalized)

    #convert RGB values (0–1) to hexadecimal
    hex_color = f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"

    #compute luminance to ensure text readability
    luminance = 0.299*r + 0.587*g + 0.114*b
    text_color = "white" if luminance < 0.5 else "black"

    #return CSS style string
    return (f"background-color:{hex_color}; color:{text_color}; padding:2px; border-radius:3px;")

In [ ]:
all_conditions = []
for rule_text in rules:
    #extract the part between IF and THEN (the conditions of the rule)
    conditions_text = rule_text.split("IF")[1].split("THEN")[0]
    
    #split multiple conditions connected by AND and remove extra spaces
    cond_list = [c.strip() for c in conditions_text.split("AND")]
    all_conditions.extend(cond_list)

#count how many times each unique condition appears across all rules
condition_freq = Counter(all_conditions)

#determine the maximum and minimum frequency for normalization in colormap
max_freq = max(condition_freq.values())
min_freq = min(condition_freq.values())

In [ ]:
#function to transform the visualization of a rule 
def rule_to_html(rule_text, condition_freq, min_freq, max_freq):
    variants = re.findall(r"Variant (\d+)", rule_text)
    variants_str = ", ".join(f"Variant {v}" for v in variants)
    conditions_text = rule_text.split("IF")[1].split("THEN")[0]
    cond_list = [c.strip() for c in conditions_text.split("AND")]

    html = f"The trace belongs to <strong>{variants_str}</strong> if: <ol style='list-style-position: inside; padding-left: 20px;'>"

    for cond in cond_list:
        #use a regular expression to parse the condition string. The pattern captures:
        #1) the feature name (.+?)
        #2) the comparison operator (<=, >=, <, >)
        #3) the threshold value (numeric)
        #4) the truth value (T or F)
        match = re.search(r"(.+?)\s*(<=|>=|<|>)\s*([\d.]+)\s*==\s*(T|F)", cond)
        if match:
            feature, operator, threshold, truth_value = match.groups()
            _, text, info = explain_condition(feature.strip(), operator, threshold, truth_value)
            parts = [] #initialize a list to store parts of the explanation for this condition
            is_positive = True #indicates whether the condition represents presence (true = green) or absence (false = red) for visualization

            #activity with 0.5
            if "activity_occurs" in info:
                if info["activity_occurs"]:
                    parts.append("does not occur in the trace")
                    is_positive = False
                else: 
                    parts.append("does occur in the trace")
                    is_positive = True
            #transition with 0.5
            if "transition_occurs" in info:
                if info["transition_occurs"]:
                    parts.append("does not occur in the trace")
                    is_positive = False
                else: 
                    parts.append("does occur in the trace")
                    is_positive = True
            #self-loops
            if "self_loop_min" in info and "self_loop_max" in info:
                min_val = info["self_loop_min"]
                max_val = info["self_loop_max"]

                if min_val == 0 and max_val == 0:
                    parts.append("is never repeated consecutively")
                    is_positive = False
                elif min_val == max_val:
                    parts.append(f"is repeated consecutively exactly {number_to_times(min_val)}")
                    is_positive = True
                elif max_val is not None:
                    parts.append(f"is repeated consecutively at most {number_to_times(max_val)}")
                    is_positive = max_val > 0
                else:
                    parts.append(f"is repeated consecutively at least {number_to_times(min_val)}")
                    is_positive = True

            #count for the activities
            if "count_min" in info or "count_max" in info:
                count_parts = []
                if "count_min" in info:
                    count_parts.append(f"occurs at least {number_to_times(info['count_min'])}")
                    is_positive = info["count_min"] > 0
                if "count_max" in info:
                    count_parts.append(f"occurs at most {number_to_times(info['count_max'])}")
                    if info["count_max"] == 0:
                        is_positive = False
                if count_parts:
                    parts.append(", and ".join(count_parts))

            freq = condition_freq.get(cond, 1)
            color = color_selector(freq, min_freq, max_freq, is_positive)
            html += f"<li style='margin-bottom:4px;'><span style='{color}'>{text} " + ", and ".join(parts) + "</span></li>"

    html += "</ol>"
    return html

In [ ]:
html_output = ""
for r in rules:
    #convert the rule into an HTML representation with colored conditions
    html_output += rule_to_html(r, condition_freq, min_freq, max_freq)
    html_output += "<hr>"

#display all translated rules as human-readable HTML with color-coded conditions
display(HTML(html_output))